# Key Use Cases — Code Clippings

Small runnable examples for the main mycontext-ai scenarios. No API key required for these cells (they use `build_context`, `transform`, `suggest_patterns`, `QualityMetrics`, and exports). Optional: run `context.execute(provider="openai")` with an API key for full LLM output.

See [docs/KEY_USE_CASES_AND_CODE_CLIPPINGS.md](../docs/KEY_USE_CASES_AND_CODE_CLIPPINGS.md) for the full document with industry pain points and key APIs.

In [1]:
from pathlib import Path
from mycontext import Context, Guidance, Directive
from mycontext.intelligence import transform, suggest_patterns, get_pattern_class, QualityMetrics, TransformationEngine
from mycontext.templates.free.decision import DecisionFramework
from mycontext.utils.structured_output import output_format
from mycontext.skills import SkillRunner, improvement_report, suggested_edits

## 1. Create a custom prompt from scratch

Structure role, rules, and task in one portable Context.

In [23]:
ctx = Context(
    guidance=Guidance(
        role="Senior security reviewer",
        rules=["Flag every injection risk", "Suggest concrete fixes"],
        style="concise, actionable",
    ),
    directive=Directive(content="Review this API for auth and input validation."),
)
print(ctx.to_markdown() + "...")

# Context
## Guidance
**Role:** Senior security reviewer
**Rules:**
- Flag every injection risk
- Suggest concrete fixes
**Style:** concise, actionable

## Directive
Review this API for auth and input validation.
...


## 2. Turn a raw question into a perfect prompt (auto pattern)

One call selects the right cognitive pattern and builds the context.

In [6]:
context = transform("Should we migrate to microservices? Compare tradeoffs.")
print("Built context:", len(context.directive.content) if context.directive else 0, "chars")
print(context.to_markdown() + "...")

Built context: 3447 chars
# Context
## Guidance
**Role:** Expert Decision Analyst and Strategic Advisor
**Rules:**
- Define decision clearly before analyzing
- Generate comprehensive option set
- Establish explicit criteria
- Evaluate objectively with evidence
- Consider short and long-term implications
- Acknowledge uncertainty and risks
- Provide actionable recommendations
**Style:** analytical, balanced, decisive

## Directive
Apply systematic decision framework:

**DECISION**: Should we migrate to microservices? Compare tradeoffs.



**OPTIONS**: To be identified during analysis


**ANALYSIS DEPTH**: quick

Systematic decision analysis:

1. **DECISION DEFINITION**
   - Core decision: [State clearly]
   - Why now?: [What triggers this decision?]
   - Stakeholders: [Who's affected?]
   - Constraints: [What limits choices?]
   - Success definition: [What does good outcome look like?]

2. **OPTION GENERATION**
   List all viable options:
   
   - To be identified
   
   Additional Opti

## 3. Use a built-in cognitive template from scratch

Pick a pattern (e.g. DecisionFramework) and build_context with your inputs.

In [8]:
df = DecisionFramework()
ctx = df.build_context(
    decision="Choose database for new service",
    options=["Postgres", "MongoDB", "DynamoDB"],
    depth="comprehensive",
)
print("Directive length:", len(ctx.directive.content))
print(ctx.directive.content + "...")

Directive length: 3467
Apply systematic decision framework:

**DECISION**: Choose database for new service



**OPTIONS UNDER CONSIDERATION**:
1. Postgres
2. MongoDB
3. DynamoDB


**ANALYSIS DEPTH**: comprehensive

Systematic decision analysis:

1. **DECISION DEFINITION**
   - Core decision: [State clearly]
   - Why now?: [What triggers this decision?]
   - Stakeholders: [Who's affected?]
   - Constraints: [What limits choices?]
   - Success definition: [What does good outcome look like?]

2. **OPTION GENERATION**
   List all viable options:
   
   - Postgres
- MongoDB
- DynamoDB
   
   Additional Options to Consider:
   - [Any missing alternatives?]
   - [Creative solutions?]
   - [Hybrid approaches?]
   - [Do-nothing option?]

3. **CRITERIA ESTABLISHMENT**
   Decision criteria (what matters):
   
   Must-Have Criteria (Deal-breakers):
   - [Criterion 1]: [Why essential?]
   - [Criterion 2]: [Why essential?]
   
   Important Criteria (High weight):
   - [Criterion A]: [Why important?]

## 4. Auto-suggest which template(s) to use

Map a question to the best patterns and optional workflow chain.

In [9]:
result = suggest_patterns(
    "Why did churn spike? Timeline, root cause, and future scenarios.",
    suggest_chain=True,
)
print("Suggested chain:", result.suggested_chain)
print(result.to_markdown())

Suggested chain: ['temporal_sequence_analyzer', 'root_cause_analyzer', 'causal_reasoner', 'future_scenario_planner']
### Why did churn spike? Timeline, root cause, and future scenarios.

- **Source**: `keyword`
- **Suggested**: `temporal_sequence_analyzer, root_cause_analyzer, causal_reasoner, future_scenario_planner`
- **Workflow chain**: `['temporal_sequence_analyzer', 'root_cause_analyzer', 'causal_reasoner', 'future_scenario_planner']`

**Reasoning**:
- `temporal_sequence_analyzer` (step 1): Question mentions 'timeline' -> Events/sequence analysis
- `root_cause_analyzer` (step 2): Question mentions 'root cause' -> Root cause analysis
- `causal_reasoner` (step 3): Question mentions 'cause' -> Causal reasoning
- `future_scenario_planner` (step 4): Question mentions 'scenario' -> Future scenario planning


## 5. Chain multiple patterns (workflow)

Run suggested chain: each pattern's output feeds the next.

In [10]:
result = suggest_patterns("Outage last week: what happened and what futures?", suggest_chain=True)
chain = result.suggested_chain or []
prev = "Outage last week: what happened and what futures?"
for name in chain[:2]:
    cls = get_pattern_class(name)
    if not cls or not hasattr(cls(), "build_context"):
        continue
    p = cls()
    kwargs = {"problem": prev[:500]}
    if "root_cause" in name:
        kwargs["symptoms"] = prev[:1500]
    elif "causal_reasoner" in name:
        kwargs = {"phenomenon": prev[:500]}
    try:
        ctx = p.build_context(**kwargs)
    except TypeError:
        try:
            ctx = p.build_context(problem=prev[:500])
        except TypeError:
            continue
    prev = ctx.directive.content
    print(f"  {name}: {len(prev)} chars")
print("Final context length:", len(prev))

  root_cause_analyzer: 5738 chars
  future_scenario_planner: 4156 chars
Final context length: 4156


## 6. Measure prompt/context quality

Score context on six dimensions; get issues and strengths.

In [11]:
ctx = Context(
    guidance=Guidance(role="Analyst", rules=["Be precise"]),
    directive=Directive(content="Analyze Q3 sales data."),
)
metrics = QualityMetrics()
score = metrics.evaluate(ctx)
print(metrics.report(score))

Context Quality Report

Overall Score: 83.5% ✅

Dimension Scores:
  ✅ Clarity: 100.0%
  ✅ Completeness: 80.0%
  ⚠️ Specificity: 70.0%
  ⚠️ Relevance: 60.0%
  ✅ Structure: 100.0%
  ✅ Efficiency: 100.0%

Strengths (11):
  ✓ No vague language
  ✓ Clear role and directive structure
  ✓ Has guidance component
  ✓ Has directive component
  ✓ Has behavioral rules
  ✓ Highly specific language
  ✓ Well-organized sections
  ✓ Clear hierarchical structure
  ✓ Uses formatting for readability
  ✓ Very token-efficient
  ✓ Minimal redundancy

Issues (3):
  ✗ No constraints defined
  ✗ No examples or specific details
  ✗ Directive too short, may lack detail

Suggestions for Improvement:
  1. Context quality is good! Consider minor refinements based on specific use case.

Metadata:
  mode: heuristic
  assembled_length: 75
  has_guidance: True
  has_directive: True
  has_knowledge: False



## 7. Refine existing prompt with quality feedback

Use score.issues and score.suggestions to improve; compare before/after.

In [12]:
metrics = QualityMetrics()
score = metrics.evaluate(ctx)
print("Issues:", score.issues[:3] if score.issues else "None")
print("Suggestions:", score.suggestions[:3] if score.suggestions else "None")
revised = Context(
    guidance=Guidance(role="Analyst", rules=["Be precise", "Include metrics and time range"]),
    directive=Directive(content="Analyze Q3 sales data. Provide key metrics and YoY comparison."),
)
score2 = metrics.evaluate(revised)
print("Revised overall:", f"{score2.overall:.2f}")
diff = metrics.compare(ctx, revised)
print("Comparison (overall delta):", diff.get("improvement", 0))

Issues: ['No constraints defined', 'No examples or specific details', 'Directive too short, may lack detail']
Suggestions: ['Context quality is good! Consider minor refinements based on specific use case.']
Revised overall: 0.85
Comparison (overall delta): 0.015000000000000013


## 8. Enforce structured output (JSON / Markdown)

Add output_format to the directive for parseable LLM responses.

In [13]:
instruction = output_format("json", schema={"summary": "str", "risks": "list", "recommendation": "str"})
ctx = Context(directive=Directive(content=f"Analyze this project proposal.\n\n{instruction}"))
print(ctx.directive.content[:450] + "...")

Analyze this project proposal.


**OUTPUT FORMAT**: Respond with valid JSON only. No additional text.

**Expected JSON Schema**:
```json
{
  "summary": "str",
  "risks": "list",
  "recommendation": "str"
}
```

Ensure your response is parseable JSON matching this schema....


## 9. One context, any LLM or framework

Export the same context to OpenAI, Anthropic, LangChain, YAML.

In [14]:
context = transform("What are the main risks of this migration?")
openai_format = context.to_openai()
print("OpenAI keys:", list(openai_format.keys()))
print("YAML (first 300 chars):", context.to_yaml()[:300] + "...")

OpenAI keys: ['messages', 'temperature', 'max_tokens']
YAML (first 300 chars): guidance:
  role: Expert Question Analyst and Cognitive Scientist
  rules:
  - Break down questions systematically using structured analysis
  - Identify implicit assumptions and knowledge requirements
  - Classify questions by type and complexity
  - Provide clear reformulations that capture intent...


## 10. Explain why a pattern was chosen

Get a human-readable explanation of input analysis and pattern selection.

In [15]:
engine = TransformationEngine()
explanation = engine.explain_selection("Should we use Kubernetes or ECS?")
print(explanation)

Input Analysis for: "Should we use Kubernetes or ECS?"

Input Type: decision
Complexity: simple
Domain: general
Ambiguity Level: low

Reasoning Requirements:
- Requires reasoning: False
- Requires comparison: False
- Requires verification: False

Recommended Patterns:
1. decision_framework
2. risk_assessor

Confidence in selection: 80.0%


---

## Advanced use cases

Context chaining, Agent Skills (SKILL.md), quality-gated execution, refinement, pattern fusion, and framework integration.

### 11. Context chaining (multi-stage pipeline)

Chain enterprise patterns: each stage's output becomes the next stage's input. No LLM calls until you execute the final context.

In [24]:
from mycontext.templates.enterprise.temporal import TemporalSequenceAnalyzer, FutureScenarioPlanner
from mycontext.templates.enterprise.diagnostic import RootCauseAnalyzer
from mycontext.templates.enterprise.synthesis import HolisticIntegrator

raw = "Q3: Complaints up 40%. Q2: Competitor launched. Q1: Support tickets doubled."
ctx1 = TemporalSequenceAnalyzer().build_context(events=raw, time_span="12 months", context="SaaS decline")
s1 = ctx1.directive.content
ctx2 = RootCauseAnalyzer().build_context(problem="Satisfaction collapse", symptoms=s1[:2500])
s2 = ctx2.directive.content
ctx3 = FutureScenarioPlanner().build_context(focal_question="How will retention evolve?", time_horizon="18 months", current_situation=s2[:2000])
s3 = ctx3.directive.content
ctx4 = HolisticIntegrator().build_context(topic="Recovery strategy", perspectives=f"Temporal: {s1[:600]}...\nRCA: {s2[:600]}...\nScenarios: {s3[:600]}...")
print("Chain lengths:", len(s1), len(s2), len(s3), len(ctx4.directive.content))
print("Optional: result = ctx4.execute(provider='openai')")

Chain lengths: 4207 8133 6191 9615
Optional: result = ctx4.execute(provider='openai')


### 12. Run Agent Skills (SKILL.md) with a quality gate

Load a skill, build context, evaluate quality; execution is skipped when score is below threshold.

In [17]:
skill_dir = Path("skills/compare_options")
if not skill_dir.exists():
    skill_dir = Path("examples/skills/compare_options")
if skill_dir.exists():
    result = SkillRunner().run(skill_dir, task="REST vs GraphQL", topic="REST vs GraphQL", depth="detailed", execute=False, quality_threshold=0.7)
    print("Quality:", result.quality_score.overall, "| Gated:", result.gated)
    print(improvement_report(result)[:500])
else:
    print("Skill path not found; run from repo root or set skill_dir to your SKILL.md directory.")

Quality: 0.865 | Gated: False
# Skill quality report

**Overall score:** 0.86 (0–1)

**Skill:** Compare Options

## Issues
- Excessive ambiguous pronouns
- No constraints defined

## Strengths
- No vague language
- Clear role and directive structure
- Has guidance component
- Has directive component
- Has behavioral rules
- Detailed directive
- Includes examples or specifics
- Well-organized sections
- Clear hierarchical structure
- Uses formatting for readability

## Suggestions
- Context quality is good! Consider minor ref


### 13. Refine SKILL.md with improvement_report and suggested_edits

Get a quality report and concrete edit suggestions for skill authors.

In [18]:
skill_dir_13 = Path("skills/compare_options")
if not skill_dir_13.exists():
    skill_dir_13 = Path("examples/skills/compare_options")
if skill_dir_13.exists():
    res = SkillRunner().run(skill_dir_13, task="Compare A and B", topic="Option A vs Option B", depth="basic", execute=False)
    print(improvement_report(res))
    for e in suggested_edits(res)[:5]:
        print("- ", e)
else:
    print("Run cell 12 first with a valid skill_dir.")

# Skill quality report

**Overall score:** 0.86 (0–1)

**Skill:** Compare Options

## Issues
- Excessive ambiguous pronouns
- No constraints defined

## Strengths
- No vague language
- Clear role and directive structure
- Has guidance component
- Has directive component
- Has behavioral rules
- Detailed directive
- Includes examples or specifics
- Well-organized sections
- Clear hierarchical structure
- Uses formatting for readability

## Suggestions
- Context quality is good! Consider minor refinements based on specific use case.

## Dimensions
- clarity: 0.90
- completeness: 0.80
- specificity: 0.90
- relevance: 0.70
- structure: 1.00
- efficiency: 1.00
-  Consider: Context quality is good! Consider minor refinements based on specific use case.
-  Address issue: Excessive ambiguous pronouns
-  Address issue: No constraints defined


### 14. Pattern fusion: anchor a skill to a cognitive template

Build a context from a skill whose frontmatter sets pattern: comparative_analyzer (or any mycontext pattern).

In [20]:
skill_dir_14 = Path("skills/compare_options")
if not skill_dir_14.exists():
    skill_dir_14 = Path("examples/skills/compare_options")
if skill_dir_14.exists():
    ctx = Context.from_skill(skill_dir_14, topic="Postgres vs MongoDB", depth="detailed")
    print("Context built via pattern fusion:", len(ctx.directive.content), "chars")
    print(ctx.directive.content[:400] + "...")
else:
    print("Run from repo root so skills/compare_options exists.")

Context built via pattern fusion: 1348 chars
Conduct a systematic comparison of these options:

OPTIONS TO COMPARE:


CONTEXT:
Use this skill to compare alternatives fairly and consistently.

## What to provide
- **topic**: The options or decision (e.g. "REST vs GraphQL", "Tool A, Tool B, Tool C")
- **depth**: Level of analysis — `basic`, `detailed`, or `comprehensive`

## Output
You will receive a structured comparison including overview, c...


### 15. Hybrid pattern suggestion (keyword + LLM)

Merge keyword and LLM suggestions; requires OPENAI_API_KEY for mode='hybrid'.

In [21]:
r = suggest_patterns("Why did revenue drop? Timeline, root cause, next steps.", suggest_chain=True, mode="keyword")
print("Keyword:", r.suggested_chain)
try:
    r_hybrid = suggest_patterns("Why did revenue drop? Timeline, root cause, next steps.", mode="hybrid", llm_provider="openai", suggest_chain=True)
    print("Hybrid:", r_hybrid.suggested_chain, "| source:", r_hybrid.source)
except Exception as e:
    print("Hybrid (needs API key):", e)

Keyword: ['temporal_sequence_analyzer', 'root_cause_analyzer', 'causal_reasoner']
Hybrid: ['temporal_sequence_analyzer', 'root_cause_analyzer', 'causal_reasoner'] | source: hybrid


### 16. Drop-in framework integration (e.g. LangChain)

Export a context to LangChain messages (or other frameworks).

In [22]:
from mycontext.integrations import LangChainHelper

context = transform("Top 3 risks for this launch?")
lc_messages = LangChainHelper.to_messages(context)
print("LangChain messages:", len(lc_messages), "items")
m0 = lc_messages[0]
print("First message type:", type(m0).__name__, "| content length:", len(getattr(m0, "content", str(m0))))

LangChain messages: 1 items
First message type: SystemMessage | content length: 3092


---

**Next steps:** Run with your own questions; try `context.execute(provider='openai')` with an API key. See [docs/KEY_USE_CASES_AND_CODE_CLIPPINGS.md](../docs/KEY_USE_CASES_AND_CODE_CLIPPINGS.md) for the full document including advanced use cases.